In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
%%capture
try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
except: _numpy = "numpy"; _pil = "pillow"
!uv pip install -qqq \
    "torch>=2.8.0" "triton>=3.4.0" {_numpy} {_pil} torchvision bitsandbytes \
    unsloth "unsloth_zoo>=2026.4.6" transformers==5.5.0 torchcodec timm

In [2]:
# %%capture
# import os, re
# if "COLAB_" not in "".join(os.environ.keys()):
#     # Local installation
#     !pip install unsloth
# else:
#     # Colab installation
#     import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
#     xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
#     !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
#     !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
# !pip install --no-deps transformers==5.5.0
# !pip uninstall huggingface_hub -y

# !pip install huggingface_hub
# !pip install -U diffusers transformers

import torch; torch._dynamo.config.cache_size_limit = 64;  # Or your desired limit

In [3]:
from unsloth import FastModel
import torch

# Configuration
MODEL_NAME = "unsloth/gemma-4-E2B-it"
MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = True  # Use False for 16-bit LoRA (more accurate but more VRAM)

model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,  # Auto detect
    load_in_4bit=LOAD_IN_4BIT,
    full_finetuning=False,
    device_map = "balanced"
    # token="YOUR_HF_TOKEN",  # Uncomment if needed
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/ashu/github/MedLens/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.4: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 4070 Ti SUPER. Num GPUs = 1. Max memory: 15.992 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 2011/2011 [01:32<00:00, 21.83it/s] 


In [4]:
# Add LoRA adapters
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers=False,   # Text only - no vision
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=32,                # LoRA rank - larger = higher accuracy but might overfit
    lora_alpha=32,       # Recommended alpha == r at least
    lora_dropout=0,
    bias="none",
    random_state=3407,
)

In [4]:
# Apply Gemma-4 chat template
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template="gemma-4",  # Use "gemma-4-thinking" if you want thinking mode
)

In [5]:
import json
from datasets import load_dataset

# For Colab: upload the file or mount drive
# DATASET_PATH = r"/kaggle/input/datasets/ashutoshmishra1998/medlensv1-2026-04-14/medlens_train.jsonl"  # Adjust path as needed
DATASET_PATH = "medlens_train.jsonl"

# # Optional: Upload from your machine (Colab only)
# # from google.colab import files
# # uploaded = files.upload()

# # Optional: Mount Google Drive
# # from google.colab import drive
# # drive.mount('/content/drive')
# # DATASET_PATH = "/content/drive/MyDrive/medlens_train.jsonl"

# def load_medlens_dataset(path):
#     """Load JSONL dataset."""
#     data = []
#     with open(path, "r", encoding="utf-8") as f:
#         for line in f:
#             item = json.loads(line.strip())
#             data.append(item)
#     print(f"Loaded {len(data)} training examples")
#     return data

# raw_data = load_medlens_dataset(DATASET_PATH)


raw_data = load_dataset("json", data_files=DATASET_PATH, split="train")
print(f"Loaded {len(raw_data)} examples (Arrow-backed, not in RAM)")

Generating train split: 943197 examples [00:16, 55855.17 examples/s] 


Loaded 943197 examples (Arrow-backed, not in RAM)


In [6]:
# Look at a sample
print("Sample training example:")
print(json.dumps(raw_data[0], indent=2))

Sample training example:
{
  "messages": [
    {
      "role": "user",
      "content": "Review this medication regimen for interaction risk. Patient: a 54 year-old male. Medications: azathioprine, cyclosporine, mycophenolate mofetil, prednisone. Indications: Immunosuppressant drug therapy, Immunosuppression, Product used for unknown indication. Symptoms: Asthenia, Confusional state, Glioblastoma multiforme. Give the likely interaction severity and why it stands out."
    },
    {
      "role": "assistant",
      "content": "MAJOR interaction risk. This regimen matches a major multi-drug adverse-event signal in FAERS. Reported events: Asthenia, Confusional state, Glioblastoma multiforme. Regimen reviewed: azathioprine, cyclosporine, mycophenolate mofetil, prednisone. FAERS supports the regimen-level signal across 6 possible drug pairs, but it does not isolate one confirmed causal pair. Urgent medication review is warranted."
    }
  ]
}


In [7]:
from datasets import load_dataset

SPLIT_SEED = 3407
EVAL_RATIO = 0.10
MAX_SEQ_LENGTH = 2048
SAVE_PATH = "medlens_tokenized"

raw_dataset = raw_data

def tokenize_example(example):
    messages = example.get("messages")
    if not messages or not isinstance(messages, list):
        return {"input_ids": [], "attention_mask": []}
    try:
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
    except Exception:
        return {"input_ids": [], "attention_mask": []}
    if not isinstance(text, str):
        return {"input_ids": [], "attention_mask": []}
    text = text.removeprefix("<bos>").strip()
    if not text:
        return {"input_ids": [], "attention_mask": []}
    tok = tokenizer(text=text, truncation=True, max_length=MAX_SEQ_LENGTH)
    return {"input_ids": tok["input_ids"], "attention_mask": tok["attention_mask"]}

# Tokenize one example at a time (not batched) — no subprocess crashes
tokenized = raw_dataset.map(
    tokenize_example,
    remove_columns=raw_dataset.column_names,
    desc="Tokenizing",
)

# Drop empties (skipped examples)
tokenized = tokenized.filter(lambda x: len(x["input_ids"]) > 0)

# Split
dataset_split = tokenized.train_test_split(
    test_size=EVAL_RATIO, seed=SPLIT_SEED, shuffle=True
)
train_dataset = dataset_split["train"]
eval_dataset = dataset_split["test"]

# Save to disk so you never redo this
train_dataset.save_to_disk(f"{SAVE_PATH}/train")
eval_dataset.save_to_disk(f"{SAVE_PATH}/eval")

print(f"Train: {len(train_dataset)} | Eval: {len(eval_dataset)}")
print(f"Saved to {SAVE_PATH}/ — next time just load from disk")

Tokenizing:   4%|▍         | 36153/943197 [01:48<45:17, 333.79 examples/s]  


KeyboardInterrupt: 

In [11]:
from datasets import load_from_disk

# If resuming, load from disk instead of re-tokenizing:
# train_dataset = load_from_disk("medlens_tokenized/train")
# eval_dataset = load_from_disk("medlens_tokenized/eval")

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=SFTConfig(
        dataset_kwargs={"skip_prepare_dataset": True},  # already tokenized
        dataloader_num_workers=0,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        warmup_ratio=0.03,
        max_steps=-1,
        num_train_epochs=2,
        learning_rate=2e-5,
        logging_steps=100,
        eval_strategy="steps",
        eval_steps=1000,
        save_strategy="steps",
        save_steps=20000,
        optim="adamw_8bit",
        weight_decay=0.001,
        lr_scheduler_type="cosine",
        max_grad_norm=0.3,
        seed=3407,
        output_dir="outputs",
        report_to="none",
        fp16=True,
        bf16=False,
    ),
)

Skipped 0 bad samples


Filter:   0%|          | 0/943197 [00:00<?, ? examples/s]

Formatted 943197 examples
Train: 848877 | Eval: 94320
Tokenizer sanity check complete
Sample token lengths: min=151, max=243
{'text': '<|turn>user\nI have been dealing with Transient ischaemic attack. I am a 74 year-old female. My medications are acetaminophen, aspirin dl-lysine, atorvastatin, desloratadine, peginterferon alfa-2a. I take them for Hypercholesterolaemia, Pain, Product used for unknown indication. Which interaction pattern is most concerning here?<turn|>\n<|turn>model\nMAJOR interaction risk. This regimen matches a major multi-drug adverse-event signal in FAERS. Reported events: Transient ischaemic attack. Regimen reviewed: acetaminophen, aspirin dl-lysine, atorvastatin, desloratadine, peginterferon alfa-2a. FAERS supports the regimen-level signal across 10 possible drug pairs, but it does not isolate one confirmed causal pair. Urgent medication review is warranted.<turn|>'}


In [12]:
# Preview formatted training text
print("Formatted train text sample:")
print("-" * 50)
print(train_dataset[0]["text"][:1000])
print("-" * 50)

print("\nFormatted eval text sample:")
print("-" * 50)
print(eval_dataset[0]["text"][:1000])
print("-" * 50)

Formatted train text sample:
--------------------------------------------------
<|turn>user
I have been dealing with Transient ischaemic attack. I am a 74 year-old female. My medications are acetaminophen, aspirin dl-lysine, atorvastatin, desloratadine, peginterferon alfa-2a. I take them for Hypercholesterolaemia, Pain, Product used for unknown indication. Which interaction pattern is most concerning here?<turn|>
<|turn>model
MAJOR interaction risk. This regimen matches a major multi-drug adverse-event signal in FAERS. Reported events: Transient ischaemic attack. Regimen reviewed: acetaminophen, aspirin dl-lysine, atorvastatin, desloratadine, peginterferon alfa-2a. FAERS supports the regimen-level signal across 10 possible drug pairs, but it does not isolate one confirmed causal pair. Urgent medication review is warranted.<turn|>
--------------------------------------------------

Formatted eval text sample:
--------------------------------------------------
<|turn>user
Review this med

In [ ]:
# -----------------------------
# 3. TRAINER SETUP
# -----------------------------
from trl import SFTTrainer, SFTConfig

MAX_STEPS = -1
NUM_TRAIN_EPOCHS = 2
LEARNING_RATE = 2e-5

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=SFTConfig(
        dataset_text_field="text",
        dataset_num_proc=None,      # or just omit this line
        dataloader_num_workers=0,   # fully single-process loading
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        warmup_ratio=0.03,
        max_steps=-1,
        num_train_epochs=2,
        learning_rate=2e-5,
        logging_steps=100,
        eval_strategy="steps",
        eval_steps=1000,
        save_strategy="steps",
        save_steps=20000,
        optim="adamw_8bit",
        weight_decay=0.001,
        lr_scheduler_type="cosine",
        max_grad_norm=0.3,
        seed=3407,
        output_dir="outputs",
        report_to="none",
        max_length=MAX_SEQ_LENGTH,
        fp16=True,
        bf16=False,
    ),
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/848877 [00:00<?, ? examples/s]

In [ ]:
# # -----------------------------
# # 3. TRAINER SETUP
# # -----------------------------
# from trl import SFTTrainer, SFTConfig

# MAX_STEPS = -1
# NUM_TRAIN_EPOCHS = 2
# LEARNING_RATE = 2e-5

# trainer = SFTTrainer(
#     model=model,
#     tokenizer=tokenizer,
#     train_dataset=train_dataset,
#     eval_dataset=eval_dataset,
#     dataset_text_field=None,      #IMPORTANT (already tokenized)
#     dataset_num_proc=1,           #prevent internal crashes
#     args=SFTConfig(
#         per_device_train_batch_size=2,
#         gradient_accumulation_steps=8,
#         warmup_ratio=0.03,        #fixed
#         max_steps=MAX_STEPS,
#         num_train_epochs=NUM_TRAIN_EPOCHS,
#         learning_rate=LEARNING_RATE,
#         logging_steps=100,
#         eval_strategy="steps",
#         eval_steps=1000,
#         save_strategy="steps",
#         save_steps=20000,
#         optim="adamw_8bit",
#         weight_decay=0.001,
#         lr_scheduler_type="cosine",
#         max_grad_norm=0.3,
#         seed=3407,
#         output_dir="outputs",
#         report_to="none",
#         max_length=MAX_SEQ_LENGTH,
#         fp16=True,   # good for T4
#         bf16=False,
#     ),
# )

# print("Trainer ready 🚀")

In [ ]:
# Train only on assistant responses (not user inputs)
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|turn>user\n",
    response_part="<|turn>model\n",
)